# Atividade Prática 2 - Prevendo Sobreviventes do Titanic com Classificação

**Aluno:** francivaldo

Este notebook segue o roteiro da atividade opcional, com limpeza de dados, treinamento de dois modelos de classificação, avaliação e interpretação dos resultados.

## 1) Importar bibliotecas e configurar o ambiente

In [ ]:
# Bibliotecas para manipulacao de dados
import pandas as pd
import numpy as np

# Bibliotecas para visualizacao de graficos
import matplotlib.pyplot as plt
import seaborn as sns

# Ferramentas do scikit-learn para treino e avaliacao
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Melhora a visualizacao das tabelas no notebook
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.4f}".format)

# Valor fixo para que os resultados sejam reproduziveis
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Estilo padrao para os graficos
sns.set(style="whitegrid")

## 2) Carregar os dados do Titanic

In [ ]:
# Le o arquivo CSV para um DataFrame (tabela do pandas)
df_raw = pd.read_csv("titanic.csv")

# No arquivo, a coluna alvo vem como '2urvived'; aqui padronizamos para 'Survived'
df_raw = df_raw.rename(columns={"2urvived": "Survived"})

# Shape = (linhas, colunas). Bom para conferir se a leitura deu certo.
print("Dimensoes:", df_raw.shape)

# Exibe o nome das colunas para entender quais variaveis temos disponiveis
print("\nColunas:", list(df_raw.columns))

# Mostra as 5 primeiras linhas da base
# Isso ajuda a reconhecer o formato dos dados antes de limpar e modelar.
df_raw.head()

## 3) Inspecionar estrutura e valores ausentes

In [ ]:
# info() mostra tipos das colunas e quantos valores nao nulos existem em cada uma
# Serve para detectar rapidamente colunas com dados faltantes.
df_raw.info()

# describe(include='all') resume estatisticas das colunas numericas e categoricas
display(df_raw.describe(include="all"))

# isnull().sum() conta quantos valores ausentes existem por coluna
# sort_values organiza da maior para a menor quantidade de ausentes.
print("\nValores ausentes por coluna:")
print(df_raw.isnull().sum().sort_values(ascending=False))

## 4) Explorar sobrevivência por variáveis principais

In [ ]:
# Criamos uma grade com 4 graficos para uma analise exploratoria inicial
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# 1) Quantos sobreviveram e quantos nao sobreviveram?
sns.countplot(data=df_raw, x="Survived", ax=axes[0, 0])
axes[0, 0].set_title("Distribuicao de sobrevivencia")

# 2) Sobrevivencia separada pelo codigo de sexo
sns.countplot(data=df_raw, x="Sex", hue="Survived", ax=axes[0, 1])
axes[0, 1].set_title("Survived por Sex")

# 3) Sobrevivencia por classe do bilhete
sns.countplot(data=df_raw, x="Pclass", hue="Survived", ax=axes[1, 0])
axes[1, 0].set_title("Survived por Pclass")

# 4) Distribuicao de idade entre sobreviventes e nao sobreviventes
sns.boxplot(data=df_raw, x="Survived", y="Age", ax=axes[1, 1])
axes[1, 1].set_title("Age por Survived")

# Ajusta espacos para os elementos nao ficarem sobrepostos
plt.tight_layout()
plt.show()

# Media de Survived por grupo de Sex = taxa de sobrevivencia em porcentagem
print("\nTaxa de sobrevivencia por sexo (codigo bruto):")
print((df_raw.groupby("Sex")["Survived"].mean() * 100).round(2))

## 5) Selecionar colunas e preparar atributos

In [ ]:
# Fazemos uma copia para preservar o dataset original sem alteracoes
df = df_raw.copy()

# Identifica colunas criadas como placeholder ('zero', 'zero.1', etc.)
# Essas colunas nao carregam informacao util para o modelo.
zero_cols = [col for col in df.columns if col.startswith("zero")]

# Remove colunas sem utilidade para previsao
df = df.drop(columns=zero_cols)

# Passengerid e apenas um identificador do registro (nao ajuda a prever)
df = df.drop(columns=["Passengerid"], errors="ignore")

print("Colunas removidas:", zero_cols + ["Passengerid"])
print("Shape apos limpeza inicial:", df.shape)

df.head()

## 6) Tratar valores ausentes e codificar categorias

In [ ]:
# Preenche idades ausentes com a mediana (robusta a valores extremos)
df["Age"] = df["Age"].fillna(df["Age"].median())

# Para Embarked, usamos a moda (valor mais frequente)
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Guardamos uma copia antes do encoding para analises descritivas depois
df_before_encoding = df.copy()

# Converte variaveis categoricas em colunas binarias (One-Hot Encoding)
# drop_first=True evita redundancia de colunas.
df = pd.get_dummies(df, columns=["Sex", "Embarked", "Pclass"], drop_first=True)

print("Valores ausentes apos tratamento:")
print(df.isnull().sum().sum())
print("\nColunas finais:", list(df.columns))

df.head()

## 7) Separar dados de treino e teste

In [ ]:
# X = variaveis de entrada (features), y = variavel alvo (target)
X = df.drop("Survived", axis=1)
y = df["Survived"]

# Divide em treino (80%) e teste (20%)
# stratify=y mantem a proporcao de classes parecida nos dois conjuntos.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Proporcao classe 1 no treino:", y_train.mean().round(4))
print("Proporcao classe 1 no teste:", y_test.mean().round(4))

In [ ]:
# Padroniza as variaveis para media 0 e desvio padrao 1
# Isso ajuda modelos sensiveis a escala e mantem comparacao justa.
scaler = StandardScaler()

# fit_transform aprende os parametros no treino e aplica no treino
X_train_scaled = scaler.fit_transform(X_train)

# No teste usamos apenas transform para evitar vazamento de dados
X_test_scaled = scaler.transform(X_test)

print("Dados escalados prontos.")

## 8) Treinar modelo de classificação (Modelo 1: Decision Tree)

In [ ]:
# Modelo 1: Arvore de Decisao
# max_depth=5 limita a complexidade para reduzir overfitting.
tree = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)

# Treina o modelo com os dados de treino
tree.fit(X_train_scaled, y_train)

# Faz previsoes no conjunto de teste
y_pred_tree = tree.predict(X_test_scaled)

# Salva relatorio em dicionario para reutilizar na tabela comparativa
report_tree = classification_report(y_test, y_pred_tree, output_dict=True)

# Matriz de confusao: [[VN, FP], [FN, VP]]
cm_tree = confusion_matrix(y_test, y_pred_tree)

print("Acuracia da Arvore:", accuracy_score(y_test, y_pred_tree))
print("\nRelatorio de classificacao - Arvore:\n")
print(classification_report(y_test, y_pred_tree))
print("Matriz de confusao - Arvore:\n", cm_tree)

# Importancia das features: quanto cada variavel influenciou as decisoes da arvore
importancias_tree = (
    pd.DataFrame({"feature": X.columns, "importancia": tree.feature_importances_})
    .sort_values("importancia", ascending=False)
)

display(importancias_tree.head(10))

## 9) Avaliar desempenho do modelo e obter probabilidades

In [ ]:
# predict_proba retorna probabilidades para cada classe [classe_0, classe_1]
probabilidades_tree = tree.predict_proba(X_test_scaled)

print("Probabilidades (5 primeiras amostras):")
print(probabilidades_tree[:5])

# Exemplo de leitura: [0.70, 0.30] significa 70% de chance de nao sobreviver
# e 30% de chance de sobreviver.

## 10) Treinar segundo modelo (Random Forest) e interpretar importâncias

In [ ]:
# Modelo 2: Random Forest (conjunto de varias arvores)
# Em geral, tende a ser mais robusto que uma unica arvore.
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

# Treina o modelo
rf.fit(X_train_scaled, y_train)

# Faz previsoes no teste
y_pred_rf = rf.predict(X_test_scaled)

# Salva metricas para comparacao posterior
report_rf = classification_report(y_test, y_pred_rf, output_dict=True)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Acuracia Random Forest:", accuracy_score(y_test, y_pred_rf))
print("\nRelatorio de classificacao - Random Forest:\n")
print(classification_report(y_test, y_pred_rf))
print("Matriz de confusao - Random Forest:\n", cm_rf)

# Importancia media das features no conjunto de arvores
importancias_rf = (
    pd.DataFrame({"feature": X.columns, "importancia": rf.feature_importances_})
    .sort_values("importancia", ascending=False)
)

display(importancias_rf.head(10))

## 11) Tabela resumo dos modelos

In [ ]:
# Monta uma tabela simples para comparar os dois modelos
# Focamos na classe 1 (sobreviventes), como pedido no enunciado.
resumo = pd.DataFrame(
    [
        {
            "Modelo": "Decision Tree",
            "Acuracia": accuracy_score(y_test, y_pred_tree),
            "Precisao_classe1": report_tree["1"]["precision"],
            "Recall_classe1": report_tree["1"]["recall"],
            "F1_classe1": report_tree["1"]["f1-score"],
        },
        {
            "Modelo": "Random Forest",
            "Acuracia": accuracy_score(y_test, y_pred_rf),
            "Precisao_classe1": report_rf["1"]["precision"],
            "Recall_classe1": report_rf["1"]["recall"],
            "F1_classe1": report_rf["1"]["f1-score"],
        },
    ]
)

# Organiza para facilitar leitura
resumo = resumo.sort_values("F1_classe1", ascending=False).reset_index(drop=True)
resumo

## 12) Taxa de sobrevivência por sexo (somente com pandas)

In [ ]:
# Calcula taxa de sobrevivencia por grupo de Sex usando apenas pandas
# mean() de Survived (0/1) representa diretamente a proporcao de sobreviventes.
taxa_surv_sex = (df_before_encoding.groupby("Sex")["Survived"].mean() * 100).round(2)

print("Taxa de sobrevivencia por codigo de Sex (%):")
print(taxa_surv_sex)

print("\nObservacao:")
print("No dataset usado, Sex=0 -> 12.93% e Sex=1 -> 50.00%.")
print("Isso sugere que o mapeamento real de 0/1 pode diferir da descricao textual e deve ser validado no dicionario do dataset.")

## 11) Gerar previsões para novos passageiros

In [ ]:
# Criamos exemplos ficticios de novos passageiros para testar o modelo treinado
novos_passageiros = pd.DataFrame(
    {
        "Age": [22, 38, 60],
        "Fare": [7.25, 71.28, 20.00],
        "Sex": [0, 1, 0],
        "sibsp": [1, 1, 0],
        "Parch": [0, 0, 0],
        "Pclass": [3, 1, 2],
        "Embarked": [2, 0, 1],
    }
)

# Aplicamos o mesmo preprocessamento usado no treino
# 1) One-Hot Encoding nas colunas categoricas
novos_enc = pd.get_dummies(novos_passageiros, columns=["Sex", "Embarked", "Pclass"], drop_first=True)

# 2) Garantimos as mesmas colunas do treino (ordem e nomes)
novos_enc = novos_enc.reindex(columns=X.columns, fill_value=0)

# 3) Aplicamos o mesmo scaler aprendido no treino
novos_scaled = scaler.transform(novos_enc)

# Predicao da classe e probabilidade de sobrevivencia (classe 1)
pred_novos = rf.predict(novos_scaled)
proba_novos = rf.predict_proba(novos_scaled)[:, 1]

# Monta tabela final para leitura amigavel
resultado_novos = novos_passageiros.copy()
resultado_novos["Predicao_Survived"] = pred_novos
resultado_novos["Probabilidade_Sobrevivencia"] = proba_novos.round(4)

resultado_novos

## 13) Respostas finais (atividade)

1. **Melhor modelo:** o **Random Forest** apresentou melhor desempenho geral. Acurácia de **0.7634** (vs. 0.7595 da árvore) e F1 da classe 1 de **0.52** (vs. 0.50), além de recall maior para sobreviventes (0.50 vs. 0.46).
2. **3 características mais importantes (melhor modelo):** `Age`, `Fare` e `Sex_1`.
3. **Matriz de confusão (melhor modelo):** `[[166, 28], [34, 34]]`. O modelo erra um pouco mais ao classificar sobreviventes como não sobreviventes (**34 falsos negativos**) do que o contrário (**28 falsos positivos**), indicando tendência levemente conservadora para prever sobrevivência.
4. **Taxa de sobrevivência por sexo (pandas):** no dataset usado, `Sex=0 -> 12.93%` e `Sex=1 -> 50.00%`. Esses números indicam grande diferença entre os grupos e são coerentes com a ideia histórica de prioridade de resgate para mulheres/crianças, mas o mapeamento de 0/1 deve ser confirmado no dicionário da base.
5. **Diferença entre modelos:** os resultados são próximos, o que sugere que os padrões principais já são capturados por ambos. O ganho do Random Forest indica que há alguma não linearidade/interação entre variáveis que o conjunto de árvores aproveita melhor.

## 14) Conclusão

O fluxo completo de classificação foi aplicado com sucesso. Entre os dois modelos testados, o **Random Forest** teve o melhor desempenho para prever sobreviventes do Titanic neste dataset. As variáveis mais relevantes foram **Age**, **Fare** e **Sex_1**, e os resultados mostram que ainda existe dificuldade moderada em recuperar todos os sobreviventes (falsos negativos).